# Russian LLM Pretraining and Instruction Tuning

This notebook demonstrates two complementary LLM workflows: training a compact decoder-only language model on a Russian literature corpus and instruction-tuning Qwen2.5-0.5B with LoRA on a Russian Alpaca-style dataset.

## 0. Imports and Configuration

The configuration keeps generated artifacts under `outputs/`, uses Hugging Face tooling, and adapts precision settings to the available GPU.

In [ ]:
from pathlib import Path
import gc
import inspect
import json
import math
import os
import random
import re
import shutil
import unicodedata
import zipfile

import numpy as np
import torch
from datasets import Dataset, DatasetDict, load_dataset
from tqdm.auto import tqdm


ZIP_NAME = "RussianNovels-master.zip"
PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
PRETRAIN_DIR = OUTPUT_DIR / "pretrain"
SFT_DIR = OUTPUT_DIR / "sft"
TOKENIZER_DIR = PRETRAIN_DIR / "tokenizer_bpe_3k"

for directory in [DATA_DIR, OUTPUT_DIR, PRETRAIN_DIR, SFT_DIR, TOKENIZER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Keep the Hugging Face cache near the project for portability and disk control.
os.environ.setdefault("HF_HOME", str(PROJECT_DIR / "hf_cache"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

SEED = 42
CONTEXT_LENGTH = 512
TOKENIZER_VOCAB_SIZE = 3_000

PRETRAIN_MAX_FILES = None
PRETRAIN_MAX_STEPS = None
SFT_MAX_TRAIN_EXAMPLES = 8_000
SFT_MAX_STEPS = 40

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")

BF16_AVAILABLE = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_AVAILABLE = torch.cuda.is_available() and not BF16_AVAILABLE


## 1. Pretraining Corpus

The pretraining stage unpacks a Russian novels corpus, reads text files, cleans sentence-level content, and stores the processed corpus as a Hugging Face `Dataset`.

In [ ]:
archive_path = PROJECT_DIR / ZIP_NAME
raw_dir = DATA_DIR / "raw"
extracted_dir = raw_dir / "RussianNovels-master"
corpus_dir = extracted_dir / "corpus"

if not archive_path.exists():
    raise FileNotFoundError(f"Archive not found: {archive_path}")

if not corpus_dir.exists():
    raw_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(raw_dir)
    print(f"Archive extracted to {raw_dir}")
else:
    print(f"Archive already extracted: {corpus_dir}")

txt_files = sorted(corpus_dir.glob("*.txt"))
if PRETRAIN_MAX_FILES is not None:
    txt_files = txt_files[:PRETRAIN_MAX_FILES]

total_mb = sum(path.stat().st_size for path in txt_files) / 1024 / 1024
print(f"Files found: {len(txt_files)}")
print(f"Corpus size: {total_mb:.1f} MB")
print("First files:")
for path in txt_files[:5]:
    print(" ", path.name)

### 1.1 Sentence Cleaning

The cleaning logic normalizes Unicode, removes boilerplate artifacts, filters very short fragments, and preserves high-quality sentence-like training examples.

In [ ]:
MIN_SENTENCE_CHARS = 20
MIN_SENTENCE_WORDS = 3


def read_text_safe(path: Path) -> str:
    encodings = ["utf-8", "utf-8-sig", "cp1251", "cp866", "latin-1"]
    last_error = None
    for encoding in encodings:
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise UnicodeDecodeError(
        "unknown",
        b"",
        0,
        1,
        f"Could not read {path} with the supported encodings: {last_error}",
    )


def is_cyrillic_letter(ch: str) -> bool:
    return (
        "\u0400" <= ch <= "\u04FF"
        or "\u0500" <= ch <= "\u052F"
        or ch in {"ё", "Ё"}
    )


def has_non_cyrillic_letters(text: str) -> bool:
    return any(ch.isalpha() and not is_cyrillic_letter(ch) for ch in text)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\ufeff", " ").replace("\u00a0", " ")
    text = text.replace("«", '"').replace("»", '"').replace("„", '"').replace("“", '"').replace("”", '"')
    text = re.sub(r"[‐‑‒–—―]+", "-", text)
    text = re.sub(r"_{2,}", " ", text)
    text = re.sub(r"\[[0-9IVXLCDMivxlcdm]+\]", " ", text)
    text = re.sub(r"\([0-9IVXLCDMivxlcdm]+\)", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_sentences(text: str) -> list[str]:
    # Split on sentence boundaries and long line breaks,
    # which often separate dialogue turns or headings in the source texts.
    pieces = re.split(r"(?<=[.!?…])\s+|\n{2,}", text)
    return [piece.strip() for piece in pieces if piece.strip()]


def clean_sentence(sentence: str) -> str | None:
    sentence = normalize_text(sentence)
    sentence = re.sub(r"\.{2,}", ".", sentence)
    sentence = re.sub(r"([!?.,;:])\1+", r"\1", sentence)
    sentence = re.sub(r"\s*-\s*", " - ", sentence)
    sentence = re.sub(r"\s+([,.!?;:])", r"\1", sentence)
    sentence = re.sub(r"([,.!?;:])(?=\S)", r"\1 ", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip(" -\t\n\r")

    if len(sentence) < MIN_SENTENCE_CHARS:
        return None
    if len(sentence.split()) < MIN_SENTENCE_WORDS:
        return None
    if has_non_cyrillic_letters(sentence):
        return None

    cyrillic_letters = sum(1 for ch in sentence if is_cyrillic_letter(ch))
    all_letters = sum(1 for ch in sentence if ch.isalpha())
    if all_letters == 0 or cyrillic_letters / all_letters < 0.95:
        return None

    return sentence


def preprocess_file(path: Path) -> tuple[list[str], dict[str, int]]:
    raw_text = read_text_safe(path)
    raw_sentences = split_sentences(raw_text)

    kept = []
    stats = {
        "raw_sentences": len(raw_sentences),
        "kept": 0,
        "short_or_empty": 0,
        "non_cyrillic": 0,
    }

    for raw_sentence in raw_sentences:
        normalized = normalize_text(raw_sentence)
        if has_non_cyrillic_letters(normalized):
            stats["non_cyrillic"] += 1
            continue

        cleaned = clean_sentence(normalized)
        if cleaned is None:
            stats["short_or_empty"] += 1
            continue

        kept.append(cleaned)
        stats["kept"] += 1

    return kept, stats

In [ ]:
clean_sentences_path = PRETRAIN_DIR / "clean_sentences.txt"
sentence_dataset_path = PRETRAIN_DIR / "clean_sentence_dataset"

seen = set()
cleaned_sentences = []
global_stats = {
    "raw_sentences": 0,
    "kept_before_dedup": 0,
    "duplicates": 0,
    "short_or_empty": 0,
    "non_cyrillic": 0,
}

for path in tqdm(txt_files, desc="Preprocessing novels"):
    sentences, stats = preprocess_file(path)
    global_stats["raw_sentences"] += stats["raw_sentences"]
    global_stats["kept_before_dedup"] += stats["kept"]
    global_stats["short_or_empty"] += stats["short_or_empty"]
    global_stats["non_cyrillic"] += stats["non_cyrillic"]

    for sentence in sentences:
        key = sentence.casefold()
        if key in seen:
            global_stats["duplicates"] += 1
            continue
        seen.add(key)
        cleaned_sentences.append(sentence)

clean_sentences_path.write_text("\n".join(cleaned_sentences), encoding="utf-8")

sentence_dataset = Dataset.from_dict({"sentence": cleaned_sentences})
if sentence_dataset_path.exists():
    shutil.rmtree(sentence_dataset_path)
sentence_dataset.save_to_disk(str(sentence_dataset_path))

print(json.dumps(global_stats, ensure_ascii=False, indent=2))
print(f"Unique cleaned sentences: {len(cleaned_sentences):,}")
print(f"Clean corpus file: {clean_sentences_path}")
print("\nExamples:")
for sentence in cleaned_sentences[:5]:
    print("-", sentence)

In [ ]:
import matplotlib.pyplot as plt

sentence_lengths = np.array([len(sentence) for sentence in cleaned_sentences])
word_counts = np.array([len(sentence.split()) for sentence in cleaned_sentences])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sentence_lengths, bins=60)
axes[0].set_title("Sentence length in characters")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")

axes[1].hist(word_counts, bins=60)
axes[1].set_title("Sentence length in words")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### 1.2 Chunks for Tokenizer Training and Pretraining

Before tokenization, cleaned sentences are grouped into text chunks. After tokenizer training, the chunks are packed into fixed 512-token causal language modeling blocks.

In [ ]:
CHUNK_TARGET_CHARS = 1_800
CHUNK_MIN_CHARS = 300


def make_text_chunks(sentences: list[str], target_chars: int = CHUNK_TARGET_CHARS) -> list[str]:
    chunks = []
    buffer = []
    buffer_len = 0

    for sentence in sentences:
        add_len = len(sentence) + 1
        if buffer and buffer_len + add_len > target_chars:
            chunk = " ".join(buffer).strip()
            if len(chunk) >= CHUNK_MIN_CHARS:
                chunks.append(chunk)
            buffer = []
            buffer_len = 0

        buffer.append(sentence)
        buffer_len += add_len

    if buffer:
        chunk = " ".join(buffer).strip()
        if len(chunk) >= CHUNK_MIN_CHARS:
            chunks.append(chunk)

    return chunks


text_chunks = make_text_chunks(cleaned_sentences)
chunk_dataset = Dataset.from_dict({"text": text_chunks})

chunk_dataset_path = PRETRAIN_DIR / "clean_chunk_dataset"
if chunk_dataset_path.exists():
    shutil.rmtree(chunk_dataset_path)
chunk_dataset.save_to_disk(str(chunk_dataset_path))

tokenizer_train_path = PRETRAIN_DIR / "tokenizer_train_text.txt"
tokenizer_train_path.write_text("\n".join(text_chunks), encoding="utf-8")

print(f"Chunks: {len(text_chunks):,}")
print(f"Average chunk length: {np.mean([len(x) for x in text_chunks]):.1f} characters")
print(text_chunks[0][:700])

## 2. Domain Tokenizer

A compact ByteLevel BPE tokenizer with a 3k vocabulary is trained for the Russian literature domain. Special tokens are `<unk>`, `<pad>`, `<bos>`, and `<eos>`.

In [ ]:
from tokenizers import Tokenizer
from tokenizers import decoders, models, normalizers, pre_tokenizers, processors, trainers
from transformers import PreTrainedTokenizerFast

SPECIAL_TOKENS = ["<unk>", "<pad>", "<bos>", "<eos>"]
tokenizer_json_path = TOKENIZER_DIR / "tokenizer.json"

if not tokenizer_json_path.exists():
    bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    bpe_tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC()])
    bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe_tokenizer.decoder = decoders.ByteLevel()

    bpe_trainer = trainers.BpeTrainer(
        vocab_size=TOKENIZER_VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )

    bpe_tokenizer.train([str(tokenizer_train_path)], trainer=bpe_trainer)
    bpe_tokenizer.post_processor = processors.TemplateProcessing(
        single="<bos> $A <eos>",
        pair="<bos> $A <eos> $B:1 <eos>:1",
        special_tokens=[
            ("<bos>", bpe_tokenizer.token_to_id("<bos>")),
            ("<eos>", bpe_tokenizer.token_to_id("<eos>")),
        ],
    )
    bpe_tokenizer.save(str(tokenizer_json_path))

fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(tokenizer_json_path),
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
)
fast_tokenizer.model_max_length = CONTEXT_LENGTH
fast_tokenizer.save_pretrained(str(TOKENIZER_DIR))

print(f"Vocabulary size: {len(fast_tokenizer):,}")
print("IDs:", {
    "unk": fast_tokenizer.unk_token_id,
    "pad": fast_tokenizer.pad_token_id,
    "bos": fast_tokenizer.bos_token_id,
    "eos": fast_tokenizer.eos_token_id,
})
sample = "Все мысли, которые имеют огромные последствия"
encoded = fast_tokenizer(sample)
print(encoded.input_ids[:40])
print(fast_tokenizer.decode(encoded.input_ids))

## 3. Causal LM Dataset

The tokenized dataset is prepared with fixed context windows and attention masks. `DataCollatorForLanguageModeling` creates shifted labels and masks padding tokens.

In [ ]:
def token_blocks_from_texts(texts: list[str], tokenizer: PreTrainedTokenizerFast, context_length: int) -> dict[str, list[list[int]]]:
    block_content_len = context_length - 2
    input_ids = []
    attention_masks = []
    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id

    for text in texts:
        ids = tokenizer(text, add_special_tokens=False, truncation=False, verbose=False)["input_ids"]
        if not ids:
            continue

        for start in range(0, len(ids), block_content_len):
            content_ids = ids[start : start + block_content_len]
            block = [bos_id] + content_ids + [eos_id]
            attention_mask = [1] * len(block)

            pad_len = context_length - len(block)
            if pad_len > 0:
                block = block + [pad_id] * pad_len
                attention_mask = attention_mask + [0] * pad_len

            input_ids.append(block[:context_length])
            attention_masks.append(attention_mask[:context_length])

    return {"input_ids": input_ids, "attention_mask": attention_masks}


chunk_split = chunk_dataset.train_test_split(test_size=0.02, seed=SEED)


def map_to_blocks(batch):
    return token_blocks_from_texts(batch["text"], fast_tokenizer, CONTEXT_LENGTH)


tokenized_dataset = DatasetDict({
    split_name: split_dataset.map(
        map_to_blocks,
        batched=True,
        batch_size=128,
        remove_columns=["text"],
        desc=f"Tokenizing {split_name}",
    )
    for split_name, split_dataset in chunk_split.items()
})

pretrain_dataset_path = PRETRAIN_DIR / "tokenized_lm_dataset_ctx512"
if pretrain_dataset_path.exists():
    shutil.rmtree(pretrain_dataset_path)
tokenized_dataset.save_to_disk(str(pretrain_dataset_path))

print(tokenized_dataset)
print("Example length:", len(tokenized_dataset["train"][0]["input_ids"]))

## 4. Decoder-Only Pretraining Model

The pretraining model uses `LlamaForCausalLM` with roughly 150M parameters, context length 512, 16 transformer layers, 16 attention heads, and 8 key-value heads.

In [ ]:
from transformers import DataCollatorForLanguageModeling, LlamaConfig, LlamaForCausalLM

llama_config = LlamaConfig(
    vocab_size=len(fast_tokenizer),
    max_position_embeddings=CONTEXT_LENGTH,
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    rms_norm_eps=1e-5,
    rope_theta=10_000.0,
    bos_token_id=fast_tokenizer.bos_token_id,
    eos_token_id=fast_tokenizer.eos_token_id,
    pad_token_id=fast_tokenizer.pad_token_id,
    tie_word_embeddings=False,
)

pretrain_model = LlamaForCausalLM(llama_config)
pretrain_model.config.use_cache = False

param_count = sum(parameter.numel() for parameter in pretrain_model.parameters())
trainable_count = sum(parameter.numel() for parameter in pretrain_model.parameters() if parameter.requires_grad)
print(f"Total parameters: {param_count / 1e6:.1f}M")
print(f"Trainable parameters: {trainable_count / 1e6:.1f}M")

data_collator = DataCollatorForLanguageModeling(tokenizer=fast_tokenizer, mlm=False)

### 4.1 Prompt Sampling During Evaluation

A callback records generations on fixed prompts during evaluation. This makes qualitative progress visible alongside training loss.

In [ ]:
from transformers import TrainerCallback

test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно",
]


@torch.inference_mode()
def generate_pretrain_samples(
    model,
    tokenizer,
    prompts: list[str],
    max_new_tokens: int = 80,
    temperature: float = 0.8,
    top_p: float = 0.9,
    save_path: Path | None = None,
) -> list[dict[str, str]]:
    model.eval()
    device = next(model.parameters()).device
    generations = []

    for prompt in prompts:
        prompt_ids = [tokenizer.bos_token_id] + tokenizer.encode(prompt, add_special_tokens=False)
        inputs = {
            "input_ids": torch.tensor([prompt_ids], dtype=torch.long, device=device),
            "attention_mask": torch.ones((1, len(prompt_ids)), dtype=torch.long, device=device),
        }
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )[0]
        decoded = tokenizer.decode(output_ids, skip_special_tokens=True)
        generations.append({"prompt": prompt, "generation": decoded})

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")

    for idx, item in enumerate(generations, start=1):
        print(f"\nPrompt {idx}: {item['prompt']}")
        print(item["generation"])

    return generations


class PromptGenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts, output_path: Path, max_new_tokens: int = 80):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.output_path = output_path
        self.max_new_tokens = max_new_tokens

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return

        print(f"\n=== Generation at eval step {state.global_step} ===")
        generations = generate_pretrain_samples(
            model=model,
            tokenizer=self.tokenizer,
            prompts=self.prompts,
            max_new_tokens=self.max_new_tokens,
        )

        record = {
            "global_step": int(state.global_step),
            "generations": generations,
        }
        with self.output_path.open("a", encoding="utf-8") as fp:
            fp.write(json.dumps(record, ensure_ascii=False) + "\n")

### 4.2 Trainer-Based Pretraining

The notebook uses `Trainer` with an effective batch size of 64, automatic gradient accumulation, cosine scheduling, and mixed precision when supported.

In [ ]:
from transformers import Trainer, TrainingArguments


def training_args_with_compatible_eval_strategy(**kwargs):
    signature = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in signature and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    elif "evaluation_strategy" in signature and "eval_strategy" in kwargs:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")

    filtered = {key: value for key, value in kwargs.items() if key in signature}
    return TrainingArguments(**filtered)


num_gpus = max(1, torch.cuda.device_count())
pretrain_per_device_batch_size = 4 if torch.cuda.is_available() else 1
target_effective_batch_size = 64
pretrain_grad_accum = max(
    1,
    math.ceil(target_effective_batch_size / (pretrain_per_device_batch_size * num_gpus)),
)
effective_batch_size = pretrain_per_device_batch_size * pretrain_grad_accum * num_gpus

pretrain_training_args = training_args_with_compatible_eval_strategy(
    output_dir=str(PRETRAIN_DIR / "llama_literature_150m"),
    overwrite_output_dir=True,
    num_train_epochs=1,
    max_steps=-1 if PRETRAIN_MAX_STEPS is None else PRETRAIN_MAX_STEPS,
    per_device_train_batch_size=pretrain_per_device_batch_size,
    per_device_eval_batch_size=pretrain_per_device_batch_size,
    gradient_accumulation_steps=pretrain_grad_accum,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    evaluation_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    bf16=BF16_AVAILABLE,
    fp16=FP16_AVAILABLE,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
)

print(f"per_device_train_batch_size: {pretrain_per_device_batch_size}")
print(f"gradient_accumulation_steps: {pretrain_grad_accum}")
print(f"effective train batch size: {effective_batch_size}")
print(f"bf16: {BF16_AVAILABLE}, fp16: {FP16_AVAILABLE}")

pretrain_generation_log = PRETRAIN_DIR / "prompt_generations_eval.jsonl"
if pretrain_generation_log.exists():
    pretrain_generation_log.unlink()

pretrain_trainer = Trainer(
    model=pretrain_model,
    args=pretrain_training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    callbacks=[
        PromptGenerationCallback(
            tokenizer=fast_tokenizer,
            prompts=test_prompts,
            output_path=pretrain_generation_log,
        )
    ],
)

pretrain_final_dir = PRETRAIN_DIR / "llama_literature_150m_final"
pretrain_train_result = pretrain_trainer.train()
pretrain_trainer.save_model(str(pretrain_final_dir))
fast_tokenizer.save_pretrained(str(pretrain_final_dir))
print(pretrain_train_result)


### 4.3 Final Pretraining Samples

The final cell saves prompt completions from the trained model so language quality can be inspected after the run.

In [ ]:
pretrain_final_generations_path = PRETRAIN_DIR / "prompt_generations_final.json"

_ = generate_pretrain_samples(
    model=pretrain_trainer.model,
    tokenizer=fast_tokenizer,
    prompts=test_prompts,
    max_new_tokens=100,
    save_path=pretrain_final_generations_path,
)

for name in ["pretrain_trainer", "pretrain_model"]:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. Supervised Fine-Tuning

The second stage fine-tunes `Qwen/Qwen2.5-0.5B` on `d0rj/alpaca-cleaned-ru`. Examples are converted into a chat-style supervised dataset.

In [ ]:
SFT_MODEL_ID = "Qwen/Qwen2.5-0.5B"
SFT_SYSTEM_PROMPT = "Ты полезный, честный и лаконичный русскоязычный ассистент."

questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?",
]

raw_sft_dataset = load_dataset("d0rj/alpaca-cleaned-ru", split="train")
print(raw_sft_dataset)
print(raw_sft_dataset.column_names)
print(raw_sft_dataset[0])

In [ ]:
def get_first_existing(example: dict, names: list[str], default: str = "") -> str:
    for name in names:
        value = example.get(name)
        if value is not None:
            return str(value)
    return default


def to_dialogue_example(example: dict) -> dict:
    instruction = get_first_existing(example, ["instruction", "prompt", "question"]).strip()
    extra_input = get_first_existing(example, ["input", "context"], "").strip()
    output = get_first_existing(example, ["output", "response", "answer"]).strip()

    if extra_input:
        user_message = f"{instruction}\n\nContext:\n{extra_input}"
    else:
        user_message = instruction

    messages = [
        {"role": "system", "content": SFT_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": output},
    ]

    return {
        "input": SFT_SYSTEM_PROMPT,
        "instruction": user_message,
        "output": output,
        "messages": messages,
    }


sft_dialogue_dataset = raw_sft_dataset.map(
    to_dialogue_example,
    remove_columns=raw_sft_dataset.column_names,
    desc="Converting Alpaca RU to dialogue format",
)
sft_dialogue_dataset = sft_dialogue_dataset.filter(
    lambda example: bool(example["instruction"].strip()) and bool(example["output"].strip()),
    desc="Filtering empty SFT examples",
)

if SFT_MAX_TRAIN_EXAMPLES is not None:
    sft_dialogue_dataset = (
        sft_dialogue_dataset
        .shuffle(seed=SEED)
        .select(range(min(SFT_MAX_TRAIN_EXAMPLES, len(sft_dialogue_dataset))))
    )

sft_dataset_path = SFT_DIR / "alpaca_cleaned_ru_dialogue"
if sft_dataset_path.exists():
    shutil.rmtree(sft_dataset_path)
sft_dialogue_dataset.save_to_disk(str(sft_dataset_path))

test_size = min(1_000, max(100, int(len(sft_dialogue_dataset) * 0.02)))
sft_split = sft_dialogue_dataset.train_test_split(test_size=test_size, seed=SEED)

print(sft_split)
print(sft_split["train"][0])

### 5.1 Tokenizer, Chat Template, and Dialogue Format

The notebook defines a Qwen-compatible chat template and preserves assistant-only loss for supervised fine-tuning.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

sft_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, trust_remote_code=True, use_fast=True)

additional_special_tokens = []
for token in ["<|im_start|>", "<|im_end|>"]:
    if token not in sft_tokenizer.get_vocab():
        additional_special_tokens.append(token)

if additional_special_tokens:
    sft_tokenizer.add_special_tokens({"additional_special_tokens": additional_special_tokens})

if sft_tokenizer.pad_token is None:
    sft_tokenizer.pad_token = sft_tokenizer.eos_token

QWEN_TRAINING_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "{% generation %}<|im_start|>assistant\n{{ message['content'] }}<|im_end|>{% endgeneration %}\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)
sft_tokenizer.chat_template = QWEN_TRAINING_CHAT_TEMPLATE


def dialogue_to_sft_text(example: dict) -> dict:
    text = sft_tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


sft_text_split = DatasetDict({
    split_name: dataset.map(
        dialogue_to_sft_text,
        remove_columns=dataset.column_names,
        desc=f"Applying chat template to {split_name}",
    )
    for split_name, dataset in sft_split.items()
})

sft_train_dataset = sft_split["train"]
sft_eval_dataset = sft_split["test"]

print(sft_text_split)
print(sft_text_split["train"][0]["text"][:1_200])
print("\nThe original messages columns are passed to SFTTrainer to enable assistant_only_loss.")

### 5.2 Baseline Generation Before SFT

Before training, the base model is sampled on several prompts. This creates a qualitative baseline for comparing the fine-tuned adapter.

In [ ]:
def get_sft_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


sft_torch_dtype = get_sft_dtype()
sft_model_kwargs = {
    "dtype": sft_torch_dtype,
    "trust_remote_code": True,
}
sft_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_ID, **sft_model_kwargs)
if torch.cuda.is_available():
    sft_model.to("cuda")
if additional_special_tokens:
    sft_model.resize_token_embeddings(len(sft_tokenizer))


@torch.inference_mode()
def generate_qwen_answers(
    model,
    tokenizer,
    questions: list[str],
    use_chat_template: bool,
    max_new_tokens: int = 180,
    temperature: float = 0.7,
    top_p: float = 0.9,
    save_path: Path | None = None,
) -> list[dict[str, str]]:
    model.eval()
    device = next(model.parameters()).device
    results = []

    for idx, question in enumerate(questions, start=1):
        if use_chat_template:
            prompt_text = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": SFT_SYSTEM_PROMPT},
                    {"role": "user", "content": question},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt_text = question

        inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
        eos_token_ids = [tokenizer.eos_token_id]
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in eos_token_ids:
            eos_token_ids.append(im_end_id)

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_token_ids,
        )[0]
        answer_ids = output_ids[inputs["input_ids"].shape[1] :]
        answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

        print(f"\nModel Input {idx}:")
        print(question)
        print(f"Model Output {idx}:")
        print(answer)

        results.append({"question": question, "answer": answer})

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

    return results


base_generation_path = SFT_DIR / "qwen_base_generations_before_sft.json"
_ = generate_qwen_answers(
    model=sft_model,
    tokenizer=sft_tokenizer,
    questions=questions_rus,
    use_chat_template=False,
    save_path=base_generation_path,
)

### 5.3 LoRA SFT

LoRA reduces GPU memory usage while adapting the base model. The training setup uses TRL `SFTTrainer` and saves the final adapter/tokenizer artifacts under `outputs/`.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:
    SFTConfig = None

USE_LORA = True
SFT_MAX_SEQ_LENGTH = 512
SFT_TARGET_EFFECTIVE_BATCH_SIZE = 8
SFT_PER_DEVICE_BATCH_SIZE = 2 if torch.cuda.is_available() else 1
SFT_GRAD_ACCUM = max(
    1,
    math.ceil(SFT_TARGET_EFFECTIVE_BATCH_SIZE / (SFT_PER_DEVICE_BATCH_SIZE * max(1, torch.cuda.device_count()))),
)

SFT_FINAL_DIR = SFT_DIR / "qwen25_05b_alpaca_ru_sft_final"

if hasattr(sft_model, "gradient_checkpointing_enable"):
    sft_model.gradient_checkpointing_enable()
sft_model.config.use_cache = False

peft_config = None
if USE_LORA:
    from peft import LoraConfig

    peft_config = LoraConfig(
        task_type="CAUSAL_LM",
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        modules_to_save=["embed_tokens", "lm_head"] if additional_special_tokens else None,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )


def build_sft_args():
    args_cls = SFTConfig if SFTConfig is not None else TrainingArguments
    signature = inspect.signature(args_cls.__init__).parameters

    kwargs = {
        "output_dir": str(SFT_DIR / "qwen25_05b_alpaca_ru_sft"),
        "overwrite_output_dir": True,
        "num_train_epochs": 1,
        "max_steps": -1 if SFT_MAX_STEPS is None else SFT_MAX_STEPS,
        "per_device_train_batch_size": SFT_PER_DEVICE_BATCH_SIZE,
        "per_device_eval_batch_size": SFT_PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": SFT_GRAD_ACCUM,
        "learning_rate": 2e-4 if USE_LORA else 2e-5,
        "weight_decay": 0.01,
        "warmup_steps": 2,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_torch",
        "logging_steps": 5,
        "eval_steps": 20,
        "save_steps": 20,
        "save_total_limit": 2,
        "bf16": BF16_AVAILABLE,
        "fp16": FP16_AVAILABLE,
        "gradient_checkpointing": True,
        "report_to": "none",
    }

    if "eval_strategy" in signature:
        kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in signature:
        kwargs["evaluation_strategy"] = "steps"

    if args_cls is not TrainingArguments:
        if "dataset_text_field" in signature:
            kwargs["dataset_text_field"] = "text"
        if "max_seq_length" in signature:
            kwargs["max_seq_length"] = SFT_MAX_SEQ_LENGTH
        elif "max_length" in signature:
            kwargs["max_length"] = SFT_MAX_SEQ_LENGTH
        if "packing" in signature:
            kwargs["packing"] = False
        if "assistant_only_loss" in signature:
            kwargs["assistant_only_loss"] = True

    filtered_kwargs = {key: value for key, value in kwargs.items() if key in signature}
    return args_cls(**filtered_kwargs)


sft_args = build_sft_args()

# For trl>=0.24/SFTConfig, use the built-in assistant_only_loss.
# It masks system/user tokens and keeps loss only on assistant responses.
data_collator = None


def build_sft_trainer():
    signature = inspect.signature(SFTTrainer.__init__).parameters
    kwargs = {
        "model": sft_model,
        "args": sft_args,
        "train_dataset": sft_train_dataset,
        "eval_dataset": sft_eval_dataset,
    }

    if "processing_class" in signature:
        kwargs["processing_class"] = sft_tokenizer
    elif "tokenizer" in signature:
        kwargs["tokenizer"] = sft_tokenizer

    if "dataset_text_field" in signature:
        kwargs["dataset_text_field"] = "text"
    if "max_seq_length" in signature:
        kwargs["max_seq_length"] = SFT_MAX_SEQ_LENGTH
    if "packing" in signature:
        kwargs["packing"] = False
    if "peft_config" in signature and peft_config is not None:
        kwargs["peft_config"] = peft_config
    if "data_collator" in signature and data_collator is not None:
        kwargs["data_collator"] = data_collator

    return SFTTrainer(**kwargs)


print(f"USE_LORA: {USE_LORA}")
print(f"SFT per_device_train_batch_size: {SFT_PER_DEVICE_BATCH_SIZE}")
print(f"SFT gradient_accumulation_steps: {SFT_GRAD_ACCUM}")
print(f"SFT effective batch size: {SFT_PER_DEVICE_BATCH_SIZE * SFT_GRAD_ACCUM * max(1, torch.cuda.device_count())}")

sft_trainer = build_sft_trainer()

In [ ]:
sft_train_result = sft_trainer.train()
sft_trainer.save_model(str(SFT_FINAL_DIR))
sft_tokenizer.save_pretrained(str(SFT_FINAL_DIR))
print(sft_train_result)


### 5.4 Generation After SFT

The final generation step checks whether the tuned model follows Russian instructions more consistently and produces better structured answers.

In [ ]:
trained_sft_model = sft_trainer.model
trained_sft_model.config.use_cache = True

sft_final_generation_path = SFT_DIR / "qwen_generations_after_sft.json"
_ = generate_qwen_answers(
    model=trained_sft_model,
    tokenizer=sft_tokenizer,
    questions=questions_rus,
    use_chat_template=True,
    max_new_tokens=180,
    temperature=0.7,
    top_p=0.9,
    save_path=sft_final_generation_path,
)
